In [ ]:
import sys

sys.path.insert(0,'C:\\code\\python_for_the_financial_economist\\')

"""
Magic commands
"""

%load_ext autoreload
%autoreload 2

"""
Load relevant packages
"""

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

import datetime

from scipy import stats
from scipy import optimize
from scipy.linalg import block_diag

import cvxpy as cp 

from pandas_datareader.data import DataReader
from pandas_datareader.famafrench import FamaFrenchReader, get_available_datasets

from typing import Union, List, Tuple

"""
Own packages
"""
import codelib.portfolio_optimization.risk_metrics as rm 
import codelib.portfolio_optimization.diversification as dm

from codelib.statistics.moments import corr_to_cov_matrix, cov_to_corr_matrix

from codelib.visualization.base import correlation_plot, fan_chart
from codelib.visualization.layout import DefaultStyle, default_colors
DefaultStyle();

from codelib.portfolio_optimization.mean_variance import portfolio_mean, portfolio_std, portfolio_variance

# Exercises - Week 9


## Problem 1: Variable transaction costs

Consider the asset universe defined by the correlation matrix

$$
C = \begin{pmatrix} 1.0 & 0.85 & 0.5 & 0.45 \\
0.85 & 1.0 & 0.5 & 0.45 \\
0.5 & 0.5 & 1.0 & 0.9 \\
0.45 & 0.45 & 0.9 & 1.0 \end{pmatrix}
$$

the expected returns $\mu = (3.2, 3.12, 8.0, 8.2)^\top$, and the volatilities $(5.0, 5.0, 22.0, 22.0)^\top$

Assume that the investor initially holds an equally weighted portfolio $w_i^0 = 1/4$, for $i=1,...,4$. 

__Question 1__

We can define the mean-variance problem in terms of changes to the initial weights 

\begin{align*}
	\min_{\mathbf{x}}  \quad & (\mathbf{w}^0 + \mathbf{x})^\top \boldsymbol{\Sigma} (\mathbf{w}^0 + \mathbf{x})\\
	\textrm{s.t.} \quad & \mathbf{x}^\top \mathbf{1} = 0 \\
      & (\mathbf{w}^0 + \mathbf{x})^\top \boldsymbol{\mu} \geq \delta \\
\end{align*}

Find the optimal $\mathbf{x}$ using `CVXPY` with $\delta = 0.045$

__Question 2__

Assume that it is not cost free to trade such that 

\begin{align*}
	\min_{\mathbf{x}}  \quad & (\mathbf{w}^0 + \mathbf{x})^\top \boldsymbol{\Sigma} (\mathbf{w}^0 + \mathbf{x})\\
	\textrm{s.t.} \quad & \mathbf{x}^\top \mathbf{1} + \phi(\mathbf{x})= 0 \\
     & (\mathbf{w}^0 + \mathbf{x})^\top \boldsymbol{\mu} \geq \delta \\
\end{align*}

where $\phi(\mathbf{x}) = \sum_{i=1}^N \phi_i(x_i)$ with 

$$
\phi_i(x_i) = \left\{ \begin{array}{cr} \alpha^{+} x_i, & x_i \geq 0 \\ -\alpha^- x_i, & x_i \leq 0 \end{array}\right.
$$

is the variable transaction costs.  Note that we can write 

$$
\phi_i = \alpha^{+} x_i^+ +  \alpha^- x_i^{-}
$$

where $x_i = x_i^{+} - x_i^{-}$ with $x_i^{+}, x_i^{-} \geq 0$. 

Implement the optimization with variable transaction costs when assuming $\alpha^+, \alpha^- = 0.01$. What is the consequence?


### Solution 

__Question 1__

In [ ]:
"""
Define inputs 
"""

corr_mat = np.array([[1.0, 0.85, 0.5, 0.45],
                     [0.85, 1.0, 0.5, 0.45],
                     [0.5, 0.5, 1.0, 0.9],
                     [0.45, 0.45, 0.9, 1.0]])

vols = np.array([5.0, 5.0, 22.0, 22.0])  / 100.0
mu = np.array([3.2, 3.12, 8.0, 8.2]) / 100.0

cov_mat = corr_to_cov_matrix(corr_mat, vols)


In [ ]:
"""
Define and solve problem 
"""

num_assets = len(mu)

w0 = np.ones(num_assets) / num_assets
delta = 0.045

# define optimization variable 
x = cp.Variable(num_assets)

# define constraints 
constraints = [cp.sum(x) == 0.0, (w0 + x) @ mu >= delta]

# define problem 
prob = cp.Problem(cp.Minimize(cp.quad_form(x + w0, cov_mat)),
                  constraints=constraints)

# solve problem 
prob.solve()

x_no_cost = x.value.copy()

# show x.value
x.value

In [ ]:
# show x.value
w0 + x.value

__Question 2__

In [ ]:
num_assets = len(mu)

w0 = np.ones(num_assets) / num_assets
delta = 0.045

# transaction costs 
alpha_pos = 0.01
alpha_neg = 0.01

# define optimization variable 
x = cp.Variable(num_assets)
x_pos = cp.Variable(num_assets)
x_neg = cp.Variable(num_assets)

# define constraints 
constraints = [x_pos >= 0.0,
               x_neg >= 0.0,
               cp.sum(x + alpha_pos*x_pos + alpha_neg * x_neg)   == 0.0,
               x == x_pos - x_neg, 
               (w0 + x) @ mu >= delta]

# define problem 
prob = cp.Problem(cp.Minimize(cp.quad_form(x + w0, cov_mat)),
                  constraints=constraints)

# solve problem 
prob.solve()

# show x.value
x.value

In [ ]:
prob.status

In [ ]:
# show allocation
w0 + x.value

We trade less! 

In [ ]:
turnover_no_cost = np.sum(np.abs(x_no_cost))
turnover_with_costs = np.sum(np.abs(x.value))

print(pd.Series(
    {
        "Without transaction costs": turnover_no_cost,
        "With transaction costs": turnover_with_costs,
    },
    name="Gross turnover (sum of absolute weight changes)",
))



## Problem 2: Constraints on the number of uncorrelated bets 

Consider the asset univers: Government bonds, Investment-grade bonds, High-yield bonds, Emerging markets gov. bonds, Equities (developed markets), Equities (Emerging markets), Private equity, Infrastructure, Real Estate, and Hedgefunds. 

$$
\mathbf{R} \sim N(\boldsymbol{\mu}, \boldsymbol{\Sigma})
$$

with correlation matrix 

\begin{equation*}
\mathbf{C} =   \begin{bmatrix}
1.0 &  0.6 &  0.1 &  0.3 & -0.1 & -0.1 & -0.2 & -0.1 & -0.1 & -0.1 \\
0.6 &  1.0 &  0.6 &  0.6 &  0.2 &  0.2 &  0.2 &  0.1 &  0.1 &  0.3 \\ 
0.1 &  0.6 &  1.0 &  0.7 &  0.7 &  0.6 &  0.6 &  0.4 &  0.3 &  0.7 \\
0.3 &  0.6 &  0.7 &  1.0 &  0.5 &  0.6 &  0.4 &  0.2 &  0.2 &  0.5 \\
-0.1 &  0.2 &  0.7 &  0.5 &  1.0 &  0.7 &  0.8 &  0.4 &  0.4 &  0.8 \\
-0.1 &  0.2 &  0.6 &  0.6 &  0.7 &  1.0 &  0.7 &  0.4 &  0.4 &  0.7 \\
-0.2 &  0.2 &  0.6 &  0.4 &  0.8 &  0.7 &  1.0 &  0.4 &  0.4 &  0.7 \\
-0.1 &  0.1 &  0.4 &  0.2 &  0.4 &  0.4 &  0.4 &  1.0 &  0.3 &  0.4 \\
-0.1 &  0.1 &  0.3 &  0.2 &  0.4 &  0.4 &  0.4 &  0.3 &  1.0 &  0.4 \\
-0.1 &  0.3 &  0.7 &  0.5 &  0.8 &  0.7 &  0.7 &  0.4 &  0.4 &  1.0
\end{bmatrix},
\end{equation*}

vector of volatilities 


\begin{equation*}
\mathbf{v} =   \begin{bmatrix}
0.037, 0.055, 0.119, 0.107, 0.153, 0.217, 0.204, 0.14, 0.108, 0.094
\end{bmatrix}^\top,
\end{equation*} 

and expected return
\begin{equation*}
\boldsymbol{\mu} =   \begin{bmatrix}
0.019, 0.022, 0.049, 0.043, 0.061, 0.083, 0.102, 0.056, 0.041, 0.038
\end{bmatrix}^\top 
\end{equation*} 

__Question 1__

Find the minimum-variance portfolio with the budget constraint $\mathbf{w}^\top \mathbf{1} = 1$ and the long-only constraint $\mathbf{w} \geq 0$. 

__Question 2__

A simplistic or naive view of portfolio diversification is to look at the number of constituents in the portfolios or the portfolio weights. One possibility is to use the norm of the portfolio weight vector, $\mathbf{w}$, to determine the Effective Number of Constituents (ENC)  (valid for any $\alpha > 0$ if weights are all positive)

\begin{equation*}
\text{ENC}_{\alpha} (\mathbf{w}) = \Vert \mathbf{w} \Vert_{\alpha}^{\frac{\alpha}{1 - \alpha}} = \left(\sum_{i=1}^N w_i^\alpha \right)^{\frac{1}{1-\alpha}}, \; \; \alpha > 0, \; \alpha \neq 1
\end{equation*}

where $N$ denote the number of assets. If $\alpha = 2$, the diversification measure is equal to the inverse of the [Herfindahl index](https://en.wikipedia.org/wiki/Herfindahl%E2%80%93Hirschman_index)

\begin{equation*}
\text{ENC}_{2} (\mathbf{w}) = \frac{1}{\sum_{i=1}^N w_i^2}
\end{equation*}

In the limiting case when $\alpha \to 1$, we obtain the exponential of the entropy of the distribution of the portfolio weight vector

\begin{equation*}
\text{ENC}_{1} = \exp \left( - \sum_{i=1}^N w_i \ln w_i\right)
\end{equation*}

Calculate the effective number of constituents with $\alpha=1$ and $\alpha=2$. Note the function `codelib.portfolio_optimization.diversification.cal`

__Question 3__

[Meucci (2009), "Managing Diversification"](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1358533) defines the _effective  number of uncorrelated bets_. 

The method relies on the choice of $N$ uncorrelated factors whose returns can be expressed as

\begin{equation}
    \mathbf{R}_F = \mathbf{A}^{-1} \mathbf{R}
\end{equation}

where $R$ is the vector of constituents returns and $\mathbf{A}$ is the "transition matrix" that maps
from the constituents returns to factor returns (square matrix of size $N$). $\mathbf{A}$ is assumed
invertible. Given the assumption of uncorrelated factors and invertible matrix, we can write the portfolio variance
as

\begin{align}
    \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} &= \mathbf{w}^\top (\mathbf{A}^\top)^{-1}
    \boldsymbol{\Sigma}_F (\mathbf{A})^{-1} \mathbf{w} \\
    &= \mathbf{w}^\top_F \boldsymbol{\Sigma} \mathbf{w}_F \\
    &= \sum_{i=1}^N (\sigma_{F_i} w_{F_i})^2
\end{align}

We can then define the i'th factors contribution to portfolio variance


\begin{equation}
    p_i =\frac{(\sigma_{F_i} w_{F_i})^2}{\mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}}
\end{equation}

which is positive and sum to one. The idea is then to calculate the effective number of uncorrelated bets using the
norm of the factor constributions to the total variance (basically calculating Effective Number of Constituents
using $p_i, i=1, ..., N$).

Using principal components as the uncorrelated factors, we obtain the matrix of eigenvector $\mathbf{V}$ as the "transition matrix" since 

$$
\mathbf{R}_{PCA} = \mathbf{V}^{-1} \mathbf{R}
$$

and the factor variances are known to be equal to the eigenvalues. 

Calculate the effective number of uncorrelated bets for the obtained minimum-variance portfolio and the equally weigthed  portfolio when using principal components as uncorrelated factors. 

Note that the function `codelib.portfolio_optimization.diversification.calculate_enb` is available. 


__Question 4__

[Deguest, Martellini, Meucci, "Risk Parity and Beyond—From Asset Allocation to Risk Allocation Decisions"](https://www-pm-research-com.esc-web.lib.cbs.dk/content/iijpormgmt/early/2022/02/08/jpm20221340) suggest a minimum variance portfolio where the number of uncorrelated bets are constrained to exceed a specific number. Implement the minimum-variance portfolio with a constraint on the number of uncorrelated bets. Ensure that the $ENB(\alpha=2)$ is equal to or larger than 8.

### Solution 

__Question 1__

In [ ]:
"""
Define inputs 
"""

asset_names = ["Government bonds", "Investment-grade bonds", "High-yield bonds",
               "Emerging markets gov. bonds", "Equities (developed markets)",
               "Equities (Emerging markets)", "Private equity", "Infrastructure",
               "Real Estate", "Hedgefunds"]

corr_mat = np.array([[1.00, 0.60, 0.10, 0.30, -0.10, -0.10, -0.20, -0.10, -0.10, -0.10],
                     [0.60, 1.00, 0.60, 0.60, 0.20, 0.20, 0.20, 0.10, 0.10, 0.30],
                     [0.10, 0.60, 1.00, 0.70, 0.70, 0.60, 0.60, 0.40, 0.30, 0.70],
                     [0.30, 0.60, 0.70, 1.00, 0.50, 0.60, 0.40, 0.20, 0.20, 0.50],
                     [-0.10, 0.20, 0.70, 0.50, 1.00, 0.70, 0.80, 0.40, 0.40, 0.80],
                     [-0.10, 0.20, 0.60, 0.60, 0.70, 1.00, 0.70, 0.40, 0.40, 0.70],
                     [-0.20, 0.20, 0.60, 0.40, 0.80, 0.70, 1.00, 0.40, 0.40, 0.70],
                     [-0.10, 0.10, 0.40, 0.20, 0.40, 0.40, 0.40, 1.00, 0.30, 0.40],
                     [-0.10, 0.10, 0.30, 0.20, 0.40, 0.40, 0.40, 0.30, 1.00, 0.40],
                     [-0.10, 0.30, 0.70, 0.50, 0.80, 0.70, 0.70, 0.40, 0.40, 1.00]])

vols = np.array([3.7, 5.5, 11.9, 10.7, 15.3, 21.7, 20.4, 14.0, 10.8, 9.4]) / 100.0

cov_mat = corr_to_cov_matrix(corr_mat=corr_mat, vols=vols)

mu = np.array([1.9, 2.2, 4.9, 4.3, 6.1, 8.3, 10.2, 5.6, 4.1, 3.8]) / 100

# number of assets 
num_assets = len(mu)

In [ ]:
"""
Find minimum-variance portfolio 
"""

# define common constraints 
sum_to_one_cons = {'type': 'eq',
                   'fun' : lambda x: np.sum(x) - 1.0, 
                   'jac' : lambda x: np.ones_like(x)}

no_short_cons = {'type': 'ineq',
                 'fun' : lambda x: x,
                 'jac' : lambda x: np.eye(len(x))}

# alternatively use 
bounds = [(0.0, 1.0)] * len(mu)

port_var_der = lambda w, cov_mat: 2 * w @ cov_mat


x0 = np.ones_like(mu) / len(mu)
res = optimize.minimize(portfolio_variance, x0=x0, args=(cov_mat,),
                        method='SLSQP',
                        jac=port_var_der,
                        constraints=[sum_to_one_cons, no_short_cons],  # no_short_cons,
                        options={'ftol': 1e-9, 'disp': True}) #, bounds=bounds)

w_min_var = res.x

In [ ]:
# We almost only allocate to government bonds!
w_min_var.round(2)

In [ ]:
# fix numerical small negative values
w_min_var = np.clip(w_min_var, 0, np.inf)

In [ ]:
w_min_var.sum()

**Conclusion:** The long-only minimum-variance portfolio is concentrated in government bonds (about 80%), with the remaining weight in infrastructure, real estate, and hedge funds. The low-volatility government-bond allocation dominates under the stated inputs.

__Question 2__

We calculate the effective number of constituents using $\alpha=1$ and $\alpha=2$

In [ ]:
dm.calculate_enc(w_min_var, alpha=1.0)

In [ ]:
dm.calculate_enc(w_min_var, alpha=2)

**Conclusion:** The two ENC measures both indicate a concentrated portfolio: the entropy-based ENC is about 2.00 constituents, while the inverse-Herfindahl ENC is about 1.52.

__Question 3__

In [ ]:
eq_weights = np.ones(num_assets) / num_assets

In [ ]:
# calculate eigenvalues and eigenvectors
eigvals, eigvec = np.linalg.eigh(cov_mat)

# sort 
idx = np.argsort(-eigvals)
pca_mat = eigvec[:, idx]
factor_variances = eigvals[idx]

In [ ]:
"""
Minimum-variance 
"""

dm.calculate_enb(w_min_var, transition_matrix=pca_mat,
                 factor_variances=factor_variances, alpha=2)

In [ ]:
"""
Equal weights
"""

dm.calculate_enb(eq_weights, transition_matrix=pca_mat,
                 factor_variances=factor_variances, alpha=2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(range(num_assets), pca_mat.T @ w_min_var, alpha=0.5, label="Minimum-variance")
ax.bar(range(num_assets), pca_mat.T @ eq_weights, alpha=0.5, label="Equally weighted")
ax.set_title("Factor weights")
ax.legend();

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(range(num_assets), dm.calculate_enb(w_min_var, transition_matrix=pca_mat,
                                           factor_variances=factor_variances, alpha=2)[1], alpha=0.5, label="Minimum-variance")

ax.bar(range(num_assets), dm.calculate_enb(eq_weights, transition_matrix=pca_mat,
                                           factor_variances=factor_variances, alpha=2)[1], alpha=0.5, label="Equally weighted")
ax.set_title("Factor risk contribution")
ax.legend();

**Conclusion:** Under the PCA factorization, the minimum-variance portfolio has an ENB of about 5.00, compared with about 1.07 for equal weights. In this example, the minimum-variance portfolio spreads its variance contribution across more uncorrelated factors.

__Question 4__

In [ ]:
"""
Find minimum-variance portfolio 
"""

enb_target = 8.00001

# define common constraints 
sum_to_one_cons = {'type': 'eq',
                   'fun' : lambda x: np.sum(x) - 1.0}

no_short_cons = {'type': 'ineq',
                 'fun' : lambda x: x}

number_of_bets = {'type': 'ineq',
                 'fun' : lambda x: dm.calculate_enb(x, pca_mat, factor_variances=factor_variances, alpha=2)[0] - enb_target}

# alternatively use 
bounds = [(0.0, 1.0)] * len(mu)


x0 = np.ones_like(mu) / len(mu)
res = optimize.minimize(portfolio_variance, x0=x0, args=(cov_mat,),
                        method='SLSQP',
                        constraints=[sum_to_one_cons, number_of_bets],  # no_short_cons,
                        options={'ftol': 1e-10, 'disp': True} , bounds=bounds)

w_min_var_enb = res.x

In [ ]:
w_min_var_enb.round(2)

In [ ]:
w_min_var.round(2)

In [ ]:
dm.calculate_enb(w_min_var_enb, transition_matrix=pca_mat, factor_variances=factor_variances, alpha=2)

**Conclusion:** The ENB constraint shifts the portfolio toward a more diversified risk allocation, with a higher variance than the unconstrained minimum-variance portfolio (about 0.00130 versus 0.00102). The recorded ENB is approximately 8, so it meets the boundary within solver tolerance but does not establish the requested strict condition $ENB \geq 8$; use a target slightly above 8 to ensure that (this is done currently in the solution).

## Problem 3: Simulation study of risk based strategies

__Question 1__

Calculate the mean-variance efficient portfolio with $\lambda = 5$ (using true parameters)

\begin{equation*}
    \boldsymbol{w}^* = \underset{\boldsymbol{w}}{\text{arg max }} \boldsymbol{w}^\top \boldsymbol{\mu} - \frac{\lambda}{2 } \boldsymbol{w}^\top \boldsymbol{\Sigma} \boldsymbol{w}
\end{equation*}

with the constraints

* $\boldsymbol{w} \geq 0$
* $\boldsymbol{w}^\top \boldsymbol{1} = 1$

__Question 2__

Calculate the following risk based strategies using the true parameters

* Naive risk parity
* Risk parity
* Minimum variance
* Minimum variance with norm constraints (delta of 0.2)
* Most diversified

Calculate the mean-variance trade-off for each portfolio. Also perform the calculation of the equally weighted portfolio. 

__Question 3__

Simulate the mean-variance loss for the risk based portfolios. Assume a sample size of 240. Use 10,000 simulations. 


Add the loss for the equally weighted strategy for comparison. 

What are your conclusions?

### Solution

__Question 1__

In [ ]:
"""
Define inputs
"""

asset_names = ["Government bonds", "Investment-grade bonds", "High-yield bonds",
               "Emerging markets gov. bonds", "Equities (developed markets)",
               "Equities (Emerging markets)", "Private equity", "Infrastructure",
               "Real Estate", "Hedgefunds"]

corr_mat = np.array([[1.00, 0.60, 0.10, 0.30, -0.10, -0.10, -0.20, -0.10, -0.10, -0.10],
                     [0.60, 1.00, 0.60, 0.60, 0.20, 0.20, 0.20, 0.10, 0.10, 0.30],
                     [0.10, 0.60, 1.00, 0.70, 0.70, 0.60, 0.60, 0.40, 0.30, 0.70],
                     [0.30, 0.60, 0.70, 1.00, 0.50, 0.60, 0.40, 0.20, 0.20, 0.50],
                     [-0.10, 0.20, 0.70, 0.50, 1.00, 0.70, 0.80, 0.40, 0.40, 0.80],
                     [-0.10, 0.20, 0.60, 0.60, 0.70, 1.00, 0.70, 0.40, 0.40, 0.70],
                     [-0.20, 0.20, 0.60, 0.40, 0.80, 0.70, 1.00, 0.40, 0.40, 0.70],
                     [-0.10, 0.10, 0.40, 0.20, 0.40, 0.40, 0.40, 1.00, 0.30, 0.40],
                     [-0.10, 0.10, 0.30, 0.20, 0.40, 0.40, 0.40, 0.30, 1.00, 0.40],
                     [-0.10, 0.30, 0.70, 0.50, 0.80, 0.70, 0.70, 0.40, 0.40, 1.00]])

vols = np.array([3.7, 5.5, 11.9, 10.7, 15.3, 21.7, 20.4, 14.0, 10.8, 9.4]) / 100.0

cov_mat = corr_to_cov_matrix(corr_mat=corr_mat, vols=vols)

mu = np.array([1.9, 2.2, 4.9, 4.3, 6.1, 8.3, 10.2, 5.6, 4.1, 3.8]) / 100

# number of assets
num_assets = len(mu)

# risk aversion
lam = 5.0

In [ ]:
# define function find the mean-variance optimal portfolio given the constraint
def calculate_mvo(mu: np.ndarray, cov_mat: np.ndarray, lam: float = 5.0, solver='OSQP'):

    # define number of assets
    num_assets = len(mu)

    # define portfolio weights to optimize over
    w = cp.Variable(num_assets)

    # define mean-variance objective
    objective = w @ mu - 0.5*lam * cp.quad_form(w, cov_mat)

    # define problem
    problem = cp.Problem(cp.Maximize(objective), constraints=[w>=0, cp.sum(w)==1])

    # solve problem
    problem.solve(solver=solver)

    return w.value

In [ ]:
def calculate_ceq(w: np.ndarray, mu: np.ndarray, cov_mat: np.ndarray, lam: float = 5.0):

    mu_port = portfolio_mean(w, mu)
    var_port= portfolio_variance(w, cov_mat)
    ceq = mu_port- 0.5 * lam * var_port

    return ceq

In [ ]:
def calculate_loss(mu: np.ndarray, cov_mat: np.ndarray, w: np.ndarray, lam: float = 5.0):

    """
    Calculates the mean-variance loss.

    Parameters
    ----------

    mu:
        Expected return vector.
    cov_mat:
        The covariance matrix.
    w:
        Portfolio weights.
    lam:
        Risk aversion parameter.

    Returns
    -------
    float
        Mean-variance loss.

    """

    w_true = calculate_mvo(mu, cov_mat, lam)
    mu_port_true = portfolio_mean(w_true, mu)
    var_port_true = portfolio_variance(w_true, cov_mat)
    ceq_true = mu_port_true - 0.5 * lam * var_port_true

    mu_port_est = portfolio_mean(w, mu)
    var_port_est = portfolio_variance(w, cov_mat)
    ceq_est = mu_port_est - 0.5 * lam * var_port_est

    loss = ceq_true - ceq_est

    return loss

In [ ]:
# calculate mvo portfolio
w_mvo = calculate_mvo(mu, cov_mat, lam=5.0)
df_weights = pd.DataFrame({'MVO': w_mvo}, index=asset_names)

In [ ]:
print(df_weights)

__Question 2__

In [ ]:
def calculate_naive_risk_parity(vols: np.ndarray) -> np.ndarray:

    """
    Calculate the inverse volatility weights

    Parameters
    ----------
    vols:
        Volatilities.

    Returns
    -------
    np.ndarray
        Portfolio weights.


    """

    inv_vols = 1.0 / vols
    sum_inv_vols = np.sum(inv_vols)

    w_iv = inv_vols / sum_inv_vols

    return w_iv

In [ ]:
def calculate_risk_parity(cov_mat: np.ndarray) -> np.ndarray:

    """
    Calculate risk parity portfolio weights given a covariance matrix.

    Parameters
    ----------
    cov_mat : np.ndarray
        Covariance matrix of asset returns, shape (n_assets, n_assets).

    Returns
    -------
    np.ndarray
        Risk parity portfolio weights, shape (n_assets,).

    Notes
    -----
    Solves a convex optimization problem to find portfolio weights that achieve risk parity,
    subject to positivity and entropy-like constraints.

    Examples
    --------
    >>> cov = np.array([[0.1, 0.02], [0.02, 0.08]])
    >>> calculate_risk_parity(cov)
    array([0.47213161, 0.52786839])
    """

    num_assets = cov_mat.shape[0]

    w = cp.Variable(num_assets)

    b = 1.0 / num_assets
    c = 0.1

    constraints = [w >= 0.0001,
                   cp.sum(b * cp.log(w)) >= c]

    prob = cp.Problem(cp.Minimize(0.5*cp.quad_form(w, cov_mat)), constraints=constraints)

    prob.solve()

    return w.value / w.value.sum()


def calculate_minimum_variance(cov_mat: np.ndarray) -> np.ndarray:

    num_assets = cov_mat.shape[0]

    # optimization variable
    w = cp.Variable(num_assets)

    # define constraints
    constraints = [cp.sum(w)==1.0, w>=0]

    # define problem
    prob = cp.Problem(cp.Minimize(w @ cov_mat @ w), constraints=constraints)

    # solve problem
    prob.solve()
    if prob.status == 'optimal':
        return w.value
    else:
        prob.solve(solver="SCS")
        return w.value

In [ ]:
def calculate_cc_ratio(weights: np.ndarray, cov_mat: np.ndarray):

    """
    Calculates the diversification ratio of Chouefaty and Coignard (2008)

    .. math::

        \\begin{equation}
            \\text{GLR}(w, \\Sigma) = \\frac{\\sum_{i=1}^N w_i \\sigma_i}{\\sqrt{w^{\\top} \\Sigma w}}
        \\end{equation}

    Parameters
    ----------
    weights:
        Portfolio weights.
    cov_mat:
        Covariance matrix.

    Returns
    -------
    float
        Diversification ratio.
    """

    port_std = rm.calculate_portfolio_std(weights=weights, cov_mat=cov_mat)

    vol_vec = np.sqrt(np.diag(cov_mat))
    avg_std = np.inner(weights, vol_vec)

    return avg_std / port_std

In [ ]:
def calculate_most_diversified_portfolio(cov_mat: np.ndarray, init_weights=None) -> np.ndarray:

    # define intial values
    n = cov_mat.shape[0]
    if init_weights is None:
        init_weights = np.repeat(1.0 / n, n)

    # define sum to one constraint
    eq_constraint = {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}

    # perform optimization
    res = optimize.minimize(lambda x: -calculate_cc_ratio(x, cov_mat) * 100 * 100, init_weights,
                            constraints=[eq_constraint,], bounds=[(0, 1)]*n)

    return res.x

In [ ]:
def calculate_minimum_variance_norm(cov_mat: np.ndarray, delta: float) -> np.ndarray:

    num_assets = cov_mat.shape[0]

    # optimization variable
    w = cp.Variable(num_assets)

    # define constraints
    constraints = [cp.sum(w)==1.0, w>=0,
                   cp.quad_form(w, np.eye(num_assets)) <= delta]

    # define problem
    prob = cp.Problem(cp.Minimize(w @ cov_mat @ w), constraints=constraints)

    # solve problem
    try:
        prob.solve()
    except:
        prob.solve(solver="SCS")

    return w.value

In [ ]:
"""
Calculate risk based portfolios using true parameters
"""

w_min_var = calculate_minimum_variance(cov_mat)
df_weights['MIN-VARIANCE'] = w_min_var

w_min_var_norm = calculate_minimum_variance_norm(cov_mat, 0.2)
df_weights['MIN-VARIANCE-NORM'] = w_min_var_norm

w_naive_rp = calculate_naive_risk_parity(vols)
df_weights['NAIVE-RP'] = w_naive_rp

w_rp = calculate_risk_parity(cov_mat)
df_weights['RISK-PARITY'] = w_rp

w_most_div = calculate_most_diversified_portfolio(cov_mat)
df_weights['MOST-DIV'] = w_most_div

df_weights['EQ'] = np.ones(num_assets) / num_assets

In [ ]:
print(df_weights.round(2))

In [ ]:
# apply calculate_ceq for each to the columns in df_weights
df_ceq_true = df_weights.apply(calculate_ceq, axis=0, args=(mu, cov_mat, lam))


In [ ]:
print(df_ceq_true)

The equally weighted portfolio gives higher utility than most of the heuristic strategies. 

__Question 3__

In [ ]:
"""
Simulate estimates of the mean, covariance, and correlation matrices
"""

# Set the number of simulations and sample size for each simulation
num_sim = 10_000
sample_size = 240

# Simulate the sample estimates of mean, covariance, and correlation matrices
sim_mu = stats.multivariate_normal.rvs(mean=mu, cov=cov_mat / sample_size, size=num_sim)
sim_cov_mat = stats.wishart.rvs(df = sample_size-1, scale = cov_mat / sample_size, size=num_sim)
sim_corr_mat = np.array([cov_to_corr_matrix(sim_cov_mat[i]) for i in range(num_sim)])

In [ ]:
"""
For each simulated covariance matrix, calculate the corresponding portfolio weights for different strategies
"""

sim_w_min_var = np.array([calculate_minimum_variance(sim_cov_mat[i]) for i in range(num_sim)])
sim_w_min_var_norm = np.array([calculate_minimum_variance_norm(sim_cov_mat[i], 0.2) for i in range(num_sim)])
sim_w_naive_rp = np.array([calculate_naive_risk_parity(np.sqrt(np.diag(sim_cov_mat[i]))) for i in range(num_sim)])
sim_w_rp = np.array([calculate_risk_parity(sim_cov_mat[i]) for i in range(num_sim)])
sim_w_most_div = np.array([calculate_most_diversified_portfolio(sim_cov_mat[i]) for i in range(num_sim)])

In [ ]:
"""
Calculate the loss for each portfolio strategy based on the simulated weights
"""

sim_loss_min_var = np.array([calculate_loss(mu, cov_mat, sim_w_min_var[i], lam) for i in range(num_sim)])
sim_loss_min_var_norm = np.array([calculate_loss(mu, cov_mat, sim_w_min_var_norm[i], lam) for i in range(num_sim)])
sim_loss_naive_rp = np.array([calculate_loss(mu, cov_mat, sim_w_naive_rp[i], lam) for i in range(num_sim)])
sim_loss_rp = np.array([calculate_loss(mu, cov_mat, sim_w_rp[i], lam) for i in range(num_sim)])
sim_loss_most_div = np.array([calculate_loss(mu, cov_mat, sim_w_most_div[i], lam) for i in range(num_sim)])

In [ ]:
"""
Calculate the loss for the equally weighted portfolio
"""

w_eq = np.ones(num_assets) / num_assets
loss_eq = calculate_loss(mu, cov_mat, w_eq, lam)

In [ ]:
sim_loss_rp.shape

In [ ]:
"""
Plot the distribution of losses for different portfolio strategies
"""

fig, ax = plt.subplots(figsize=(12, 6))

sns.kdeplot(sim_loss_min_var, fill=True, ax=ax, color="navy", alpha=0.5, label="Minimum Variance")
sns.kdeplot(sim_loss_min_var_norm, fill=True, ax=ax, label="Minimum Variance Norm")
sns.kdeplot(sim_loss_naive_rp, fill=True, ax=ax, label="Naive Risk Parity")
sns.kdeplot(sim_loss_rp, fill=True, ax=ax, label="Risk Parity")
sns.kdeplot(sim_loss_most_div, fill=True, ax=ax, label="Most Diversified")

ax.axvline(x=loss_eq, color='gray', linestyle='--', label="Equally Weighted")

ax.legend()
ax.set_title("Distribution of Loss")
ax.set_xlabel("Loss")
ax.set_ylabel("Density")
ax.set_xlim([0, max(sim_loss_min_var.max(), sim_loss_min_var_norm.max(), sim_loss_naive_rp.max(), sim_loss_rp.max(), sim_loss_most_div.max(), loss_eq)])

plt.tight_layout()

**Conclusion:** At $\lambda = 5$, the equally weighted portfolio is difficult to beat consistently; the risk-based strategies do not uniformly deliver lower loss across simulations. As risk aversion increases, variance receives more weight, so portfolios concentrated in low-risk assets may become more attractive. This is a qualitative implication, not demonstrated by the plot shown here. Try to rerun the simulation at a larger $\lambda$ to compare strategies. 

# References

## Articles

[Meucci (2009), "Managing Diversification"](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1358533)

[Deguest, Martellini, Meucci, "Risk Parity and Beyond—From Asset Allocation to Risk Allocation Decisions"](https://www-pm-research-com.esc-web.lib.cbs.dk/content/iijpormgmt/early/2022/02/08/jpm20221340)
